<a href="https://colab.research.google.com/github/bhagath-ac07/AI_learning/blob/main/Support_Ticket_Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch

emails = [
    "My credit card was charged twice.",
    "I cannot reset my password, the link is broken.",
    "We are interested in buying 50 licenses.",
    "Where can I download the invoice?",
    "The application crashes when I upload a PNG file.",
    "Do you offer a discount for non-profits?",
    "My subscription was renewed without my permission.",
    "I am getting a 404 error on the dashboard.",
    "Can we schedule a demo for our team?",
    "I need a refund for the last month.",
    "The API is returning a 500 status code.",
    "What is the pricing for the Enterprise tier?",
    "Please update the billing address on my account.",
    "The login page is not loading on Safari.",
    "I want to upgrade to the Pro plan immediately.",
    "The export button is grayed out and not working.",
    "I need a receipt for tax purposes.",
    "What is the cost for adding 5 more users?",
    "My mobile app freezes on the splash screen.",
    "My payment method was declined, how do I fix it?",
    "We are interested in signing a yearly contract.",
    "Cannot connect to the websocket server.",
    "Why is my bill higher than last month?",
    "Do you have a referral program for partners?"
]

# Label Mapping:
# 0 = Billing
# 1 = Tech
# 2 = Sales

# Labels corresponding to the 24 emails above
labels = [
    0, 1, 2, 0, 1, 2, 0, 1,
    2, 0, 1, 2, 0, 1, 2, 1,
    0, 2, 1, 0, 2, 1, 0, 2
]

In [4]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
inputs = tokenizer(emails, padding=True, truncation=True, return_tensors="pt")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased',
                                                            num_labels=3)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
class SupportTicketDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # We grab the item and detach it to save memory
        item = {
            key: val[idx].clone().detach()
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = SupportTicketDataset(inputs, labels)

In [8]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    logging_steps=10,
    dataloader_pin_memory=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

In [9]:
trainer.train()
new_eamil = "I want to upgrade my subscription to the Pro plan."
new_inputs = tokenizer(new_eamil, return_tensors="pt")
new_outputs = model(**new_inputs)
print(new_outputs.logits)
new_predicted_class = torch.argmax(new_outputs.logits)
print(new_predicted_class)

Step,Training Loss
10,1.019997
20,0.610515
30,0.391566


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

tensor([[-0.0466, -0.7940,  0.5122]], grad_fn=<AddmmBackward0>)
tensor(2)
